In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df_transformed = pd.read_csv('turtles_t2_l5_no_outliers.csv')
df_original = pd.read_csv('turtles_t2_l5_with_outliers.csv')

In [ ]:
# Разделите признаки и целевую переменную
X_transformed = df_transformed.drop(columns='weight')
y_transformed = df_transformed['weight']
X_original = df_original.drop(columns='weight')
y_original = df_original['weight']

# Разделите df_transformed на train и test
# Используйте test_size=0.2 и random_state=627
X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed = train_test_split(X_transformed, y_transformed, test_size=0.2, random_state=627)

# Теперь выделите те же записи для df_original
X_train_original = X_original.loc[X_train_transformed.index]
X_test_original = X_original.loc[X_test_transformed.index]
y_train_original = y_original.loc[y_train_transformed.index]
y_test_original = y_original.loc[y_test_transformed.index]

# Выведем размеры, чтобы убедиться, что всё совпадает
print("Размеры выборок (transformed):", X_train_transformed.shape, X_test_transformed.shape)
print("Размеры выборок (original):", X_train_original.shape, X_test_original.shape)

In [3]:
# Зададим группы признаков
cat_features = ['binomial_name']
num_features = [
    'shell_length', 'shell_width', 'head_length', 'head_width',
    'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',
    'circle_count', 'measure_count'
]
special_features = ['shell_crack']

In [ ]:
# Допишите код трансформера с полной обработкой
full_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 
         Pipeline(
             steps=[
                 ('imputer', SimpleImputer(strategy='most_frequent')),
                 ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
            ]
        ),
        cat_features
        ),
        ('num',
         Pipeline(
             steps=[
                 ('imputer', SimpleImputer(strategy='median')),
                 ('scaler', StandardScaler())
             ]
         ),
         num_features
        ),
        ('zero', SimpleImputer(strategy='constant', fill_value=0), special_features)
    ]
)

# Допишите код трансформера с частичной обработкой
light_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 
         Pipeline(
             steps=[
                 ('imputer', SimpleImputer(strategy='most_frequent')),
                 ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
            ]
        ),
        cat_features
        ),
        ('num', SimpleImputer(strategy='median'), num_features),
        ('zero', SimpleImputer(strategy='constant', fill_value=0), special_features)
    ]
)

# Создайте пайплайн для полной обработки
pipe_transformed = Pipeline(
    steps=[
        ('preprocessor', full_preprocessor),
        ('model', Ridge())
    ]
)

# Создайте пайплайн для частичной обработки
pipe_original = Pipeline(
    steps=[
        ('preprocessor', light_preprocessor),
        ('model', Ridge())
    ]
)

In [ ]:
# Примените трансформеры к X_train_transformed и X_train_original

X_train_transformed_features = full_preprocessor.fit_transform(X_train_transformed)
feature_names_transformed = full_preprocessor.get_feature_names_out()
X_train_transformed_new = pd.DataFrame(X_train_transformed_features, columns=feature_names_transformed)

print("Данные без выбросов после трансформации:")
print(X_train_transformed_new.head())

X_train_original_features = light_preprocessor.fit_transform(X_train_original)
feature_names_original = light_preprocessor.get_feature_names_out()
X_train_original_new = pd.DataFrame(X_train_original_features, columns=feature_names_original)

print("\nДанные с выбросами после трансформации:")
print(X_train_original_new.head())

In [ ]:
def train_eval_model(X_train, X_test, y_train, y_test, pipline):
    pipline.fit(X_train, y_train)
    prediction = pipline.predict(X_test)
    metrics = {'MAE': mae, 'RMSE': round(np.sqrt(mse), 2), 'R2': r2}
    return pipline, metrics